# Whisper-large-v3 기준 재현

참고: CLAUDE.md 3.1·5·7절 (할 일 2, 7)

목표: LibriSpeech test-clean WER **2.12%** 근처, UrbanSound8K 비음성 환각률 **99.97%** 근처 (Calm-Whisper).

**별도 Colab 세션에서 실행한다.** 먼저 소량으로 구동을 확인하고, 마지막 셀에서 전체를 돌린다.

In [ ]:
# 공통 설정: GitHub에서 코드 받기 + Drive 연결
# Drive에는 데이터·모델 캐시·결과를 둔다 (세션이 끊겨도 남음)
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/diffusion-asr-hallucination'
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/alsgur4082/diffusion-asr-hallucination.git {REPO}
sys.path.insert(0, REPO)

DRIVE = '/content/drive/MyDrive/dah'   # 데이터·캐시·결과 루트
DATA = f'{DRIVE}/data'
RESULTS = f'{DRIVE}/results'
os.environ['HF_HOME'] = f'{DRIVE}/hf_cache'  # 모델 import 전에 설정해야 적용됨
os.makedirs(RESULTS, exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q jiwer soundfile scipy datasets

In [ ]:
import time, torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from src.runners.base import DecodeConfig, TranscriptionResult, save_results

MODEL = 'openai/whisper-large-v3'
cfg = DecodeConfig(model=MODEL, language='en', temperature=0.0, beam_size=1, seed=0,
                   extra={'dtype': 'float16', 'task': 'transcribe'})
torch.manual_seed(cfg.seed)
processor = WhisperProcessor.from_pretrained(MODEL)
model = WhisperForConditionalGeneration.from_pretrained(MODEL, torch_dtype=torch.float16).to('cuda').eval()

def transcribe(paths, batch_size=16):
    from src.data.audio import load_mono_16k
    results = []
    for i in range(0, len(paths), batch_size):
        batch = paths[i:i + batch_size]
        audios = [load_mono_16k(p) for p in batch]
        feats = processor(audios, sampling_rate=16000, return_tensors='pt').input_features.to('cuda', torch.float16)
        t0 = time.time()
        with torch.inference_mode():
            ids = model.generate(feats, language=cfg.language, task='transcribe',
                                 do_sample=False, num_beams=cfg.beam_size)
        dt = (time.time() - t0) / len(batch)
        for p, text in zip(batch, processor.batch_decode(ids, skip_special_tokens=True)):
            results.append(TranscriptionResult(audio_path=p, transcript=text, elapsed_sec=dt, config=cfg))
    return results

## 1. 스모크 테스트 (LibriSpeech 5개)

In [ ]:
# 스모크 테스트용 소량 데이터 (LibriSpeech test-clean 5개)
from src.data.download import librispeech
ls_paths, ls_refs = librispeech(DATA, 'test-clean', n=5)
print(len(ls_paths), ls_paths[0], ls_refs[0])

In [ ]:
from src.metrics.basic import wer
res = transcribe(ls_paths)
for r, ref in zip(res, ls_refs):
    print('REF:', ref); print('HYP:', r.transcript); print()
# Whisper 기준 수치와 비교하려면 Whisper 영어 정규화기를 쓴다
print(wer(ls_refs, [r.transcript for r in res], normalize=processor.tokenizer.normalize))

## 2. 비음성 스모크 테스트 (UrbanSound8K 클래스별 2개 = 20개)

처음 실행하면 UrbanSound8K 전체(약 7GB)를 HF 캐시(Drive)에 받는다.

In [ ]:
from src.data.download import urbansound8k
from src.metrics.basic import hallucination_rate
us_paths, _, us_classes = urbansound8k(DATA, per_class=2)
res = transcribe(us_paths)
for r, c in zip(res, us_classes):
    print(f'{c:20s} -> {r.transcript!r}')
print(hallucination_rate([r.transcript for r in res]))

## 3. 전체 기준 재현

스모크가 정상이면 실행한다. 결과는 Drive `results/`에 설정과 함께 저장된다.

In [ ]:
import json
ls_paths, ls_refs = librispeech(DATA, 'test-clean')          # 2,620개
res_ls = transcribe(ls_paths)
save_results(res_ls, f'{RESULTS}/whisper-large-v3_librispeech-test-clean.jsonl')
m_ls = wer(ls_refs, [r.transcript for r in res_ls], normalize=processor.tokenizer.normalize)

us_paths, _, us_classes = urbansound8k(DATA)                 # 8,732개
res_us = transcribe(us_paths)
save_results(res_us, f'{RESULTS}/whisper-large-v3_urbansound8k.jsonl')
m_us = hallucination_rate([r.transcript for r in res_us])

summary = {'config': cfg.__dict__, 'librispeech_test_clean': m_ls, 'urbansound8k': m_us,
           'reference': {'wer_test_clean': 0.0212, 'hr_urbansound8k': 0.9997, 'source': 'Calm-Whisper arXiv 2505.12969'}}
json.dump(summary, open(f'{RESULTS}/whisper-large-v3_baseline_summary.json', 'w'), indent=1, default=str)
print(json.dumps(summary, indent=1, default=str))